### Import ans setUp

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ['GROQ_API_KEY']:
    print("Api key is set")

Api key is set


In [4]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0
)

### **RAG Implementation with PDF**

#### **Step 1 : Extracting text from Pdf**

In [5]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "./Docs/Lora.pdf"

loader = PyPDFLoader(pdf_path)

docs = loader.load()
docs


[Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-10-19T00:57:13+00:00', 'author': '', 'keywords': '', 'moddate': '2021-10-19T00:57:13+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': './Docs/Lora.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1'}, page_content='LORA: L OW-R ANK ADAPTATION OF LARGE LAN-\nGUAGE MODELS\nEdward Hu∗ Yelong Shen∗ Phillip Wallis Zeyuan Allen-Zhu\nYuanzhi Li Shean Wang Lu Wang Weizhu Chen\nMicrosoft Corporation\n{edwardhu, yeshe, phwallis, zeyuana,\nyuanzhil, swang, luw, wzchen }@microsoft.com\nyuanzhil@andrew.cmu.edu\n(Version 2)\nABSTRACT\nAn important paradigm of natural language processing consists of large-scale pre-\ntraining on general domain data and adaptation to particular tasks or domains. As\nwe pre-train larger models, full ﬁne-tuning, which retrains all model par

#### **Step 2 : Splitting the documents into Chuncks**

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, # chunk size
    chunk_overlap = 100 # take 100 character from previous chunk so data flow cant  be break
)

chunks = splitter.split_documents(docs)
chunks


[Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-10-19T00:57:13+00:00', 'author': '', 'keywords': '', 'moddate': '2021-10-19T00:57:13+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': './Docs/Lora.pdf', 'total_pages': 26, 'page': 0, 'page_label': '1'}, page_content='LORA: L OW-R ANK ADAPTATION OF LARGE LAN-\nGUAGE MODELS\nEdward Hu∗ Yelong Shen∗ Phillip Wallis Zeyuan Allen-Zhu\nYuanzhi Li Shean Wang Lu Wang Weizhu Chen\nMicrosoft Corporation\n{edwardhu, yeshe, phwallis, zeyuana,\nyuanzhil, swang, luw, wzchen }@microsoft.com\nyuanzhil@andrew.cmu.edu\n(Version 2)\nABSTRACT\nAn important paradigm of natural language processing consists of large-scale pre-\ntraining on general domain data and adaptation to particular tasks or domains. As\nwe pre-train larger models, full ﬁne-tuning, which retrains all model par

In [7]:
chunks[0].metadata

{'producer': 'pdfTeX-1.40.21',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2021-10-19T00:57:13+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2021-10-19T00:57:13+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': './Docs/Lora.pdf',
 'total_pages': 26,
 'page': 0,
 'page_label': '1'}

#### **Step 3 : Creating the embeddings from chunks**

In [8]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\Star\AppData\Local\Temp\ipykernel_6308\384917042.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### **Step 4: Create and Store a Embeddings in Vector Store**

### **Similarity Search**

In [9]:
from langchain_community.vectorstores import Chroma

vectoreStore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./Vector/"
)

### **Talk To LLM**

In [10]:
context = vectoreStore.similarity_search("what is Lora?", k=3)

response = model.invoke(f"what is Lora?, you can aswer using following context :{context}")
print(response.content)

Based on the provided context, LoRA stands for Low-Rank Adaptation. It is a method for adapting pre-trained models to new tasks without requiring the accumulated gradient update to weight matrices to have full-rank during adaptation.

In simpler terms, LoRA is a technique used in deep learning to fine-tune pre-trained models for specific tasks without losing their original performance. It achieves this by introducing a low-rank adaptation mechanism that allows the model to adapt to new tasks without requiring the full-rank gradient update.

The key benefits of LoRA include:

1. **Improved expressiveness**: LoRA can recover the expressiveness of full fine-tuning by setting the LoRA rank to the rank of the pre-trained weight matrices.
2. **No additional inference latency**: LoRA allows for explicit computation and storage of the adapted weights, which can be used for inference without additional latency.
3. **Flexibility**: LoRA can be used for various tasks and models, including pre-tra